In [ ]:
#script para verificar a vriancia do dataset por origem 
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Here I combine datasets with the same source (e.g., ce-*)
def combine_datasets(path, substring):
    files = [f for f in os.listdir(path) if substring in f and f.endswith('.csv')]
    
    if not files:
        print(f"No files containing the substring '{substring}' were found.")
        return None
    
    dataframe_list = []
    for file in files:
        file_path = os.path.join(path, file)
        df = pd.read_csv(file_path)
        dataframe_list.append(df)
    
    combined_df = pd.concat(dataframe_list, ignore_index=True)
    return combined_df

path = '../../datasets/serie-multivariada'
dataframes_by_source = {}

for dirs, root, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            source = '-' + file.split('-')[2] + '-'
            #print(f"Processing files with source: {source}")
            df = combine_datasets(path, source)
            
            if df is not None:
                key_name = f"{source.strip('-')}" 
                dataframes_by_source[key_name] = df
dataframes_by_source.pop('df', None) #df tem poucas linhas, entao a gente resolveu descatra isso 

In [ ]:
# correlacao entre Coeficiente de variância e nrmse da regressao

rmse_bbr =pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_bbr_rmse.csv')
rmse_bbr = rmse_bbr[~(rmse_bbr['source'] == 'df')]

protocol = 'Vazao_bbr'
rmse = rmse_bbr

correlations_by_model = {}
variances = {}

for key, value in dataframes_by_source.items():
    dataset = dataframes_by_source[key]
    variance_coef = (dataset[protocol].std() / dataset[protocol].mean()) * 100
    variances[key] = variance_coef

models = rmse.columns[1:]

for model in models:
    df_model = pd.DataFrame({
        'Source': rmse['source'],
        'RMSE': rmse[model],
        'Variance_coef': rmse['source'].map(lambda x: variances[x])  
    })
    correlation = df_model[['RMSE', 'Variance_coef']].corr().iloc[0, 1]
    correlations_by_model[model] = correlation

# Exibindo as correlações
print("Correlation between high RMSE and dataset variance for each model:")
for model, correlation in correlations_by_model.items():
    print(f"{model}: {correlation:.4f}")



In [ ]:
df_model

In [ ]:
#estatistica descritiva do nrmse por modelo

rmse_bbr =pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_bbr_rmse.csv')
rmse_bbr = rmse_bbr[~(rmse_bbr['source'] == 'df')] #retirando df pq tem poucas linhas

# Configuração para exibir mais casas decimais no pandas
pd.set_option('display.float_format', lambda x: '%.4f' % x)

rmse_melted = rmse.melt(id_vars=['source'], 
                        var_name='Model', 
                        value_name='RMSE')
# 2. Tabela com estatísticas descritivas do RMSE por modelo
rmse_stats = rmse_melted.groupby('Model')['RMSE'].agg([
    ('Média', 'mean'),
    ('Mediana', 'median'),
    ('Desvio Padrão', 'std'),
    ('Mínimo', 'min'),
    ('Máximo', 'max')
]).round(4)

print("\nEstatísticas Descritivas do NRMSE por Modelo:")
print(rmse_stats.to_string())

# # 4. Tabela de variâncias por fonte
# variances_df = pd.DataFrame.from_dict(variances, orient='index', 
#                                     columns=['Coeficiente de Variação'])
# variances_df = variances_df.round(4)

# print("\nCoeficiente de Variação por Fonte de Dados:")
# print(variances_df.to_string())
